# SignBridge — BdSL Sign Recognition Training


---
## Stage 1 — Setup & Mount

In [ ]:
!pip uninstall -y tensorflow mediapipe protobuf
!pip install -q mediapipe==0.10.14
!pip install openpyxl

In [ ]:
import mediapipe as mp
print(mp.__version__)
print('holistic' in dir(mp.solutions))

0.10.14
True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT    = '/content/drive/MyDrive/SignBD'
DRIVE_DATASET = f'{DRIVE_ROOT}/SignBD-Word_RGB/DATASET'
SPLIT_DIR     = f'{DRIVE_ROOT}/Train_Test_Split/Class_number_200/rgb'
XLSX_PATH     = f'{DRIVE_ROOT}/words_translation_gloss.xlsx'

OUTPUT_DIR    = f'{DRIVE_ROOT}/output'
KEYPOINT_DIR  = f'{OUTPUT_DIR}/keypoints'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(KEYPOINT_DIR, exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Output dir: /content/drive/MyDrive/SignBD/output


In [ ]:
import os
print(os.path.exists(DRIVE_DATASET))
print(os.listdir(DRIVE_DATASET)[:5] if os.path.exists(DRIVE_DATASET) else os.listdir(DRIVE_ROOT))

True
['virus', 'valo', 'tin', 'valobasha', 'upodesh']


In [ ]:
import os, shutil, time

LOCAL_VIDEO_DIR = '/content/videos'
DATASET_DIR = f'{LOCAL_VIDEO_DIR}/DATASET'

if not os.path.exists(DATASET_DIR):
    print('Copying videos from Drive to local disk...')
    t0 = time.time()
    shutil.copytree(DRIVE_DATASET, DATASET_DIR)
    print(f'Copied in {(time.time()-t0)/60:.1f} min')
else:
    print('Videos already on local disk')

print('DATASET dir:', DATASET_DIR)
print('Gloss folders:', len(os.listdir(DATASET_DIR)))

---
## Stage 2 — Extract Keypoints


In [ ]:
def parse_split(txt_path):
    entries = []
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            video_rel = parts[0]
            label_id  = int(parts[-1])
            gloss     = video_rel.split('/')[1]
            entries.append({
                'video_rel': video_rel,
                'gloss': gloss,
                'label_id': label_id,
            })
    return entries

train_entries = parse_split(f'{SPLIT_DIR}/train_class_200_rgb.txt')
test_entries  = parse_split(f'{SPLIT_DIR}/test_class_200_rgb.txt')

print(f'Train videos: {len(train_entries)}')
print(f'Test videos : {len(test_entries)}')


label_to_gloss = {}
for e in train_entries + test_entries:
    label_to_gloss[e['label_id']] = e['gloss']
num_classes = len(label_to_gloss)
print(f'Classes: {num_classes}')
print('Sample:', dict(list(sorted(label_to_gloss.items()))[:5]))

Train videos: 4800
Test videos : 1200
Classes: 199
Sample: {1: 'janala', 2: 'osubidha', 3: 'dhorjo', 4: 'sajano', 5: 'surjo'}


In [ ]:
import cv2
import numpy as np
import mediapipe as mp

mp_holistic = mp.solutions.holistic

POSE_UPPER = [11, 12, 13, 14, 15, 16]
N_HAND = 21
TARGET_FRAMES = 30
FEATURE_DIM = len(POSE_UPPER) * 3 + N_HAND * 3 + N_HAND * 3

_holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

def extract_frame_keypoints(results):
    vec = []
    if results.pose_landmarks:
        lms = results.pose_landmarks.landmark
        for idx in POSE_UPPER:
            lm = lms[idx]
            vec.extend([lm.x, lm.y, lm.visibility])
    else:
        vec.extend([0.0] * (len(POSE_UPPER) * 3))
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            vec.extend([lm.x, lm.y, 1.0])
    else:
        vec.extend([0.0] * (N_HAND * 3))
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            vec.extend([lm.x, lm.y, 1.0])
    else:
        vec.extend([0.0] * (N_HAND * 3))
    return np.array(vec, dtype=np.float32)

def resample_frames(frames, target=TARGET_FRAMES):
    if len(frames) == 0:
        return np.zeros((target, FEATURE_DIM), dtype=np.float32)
    frames = np.array(frames, dtype=np.float32)
    if len(frames) == target:
        return frames
    idx = np.linspace(0, len(frames) - 1, target).astype(int)
    return frames[idx]

def normalize_keypoints(kps):
    kps = kps.copy()
    l_sh = kps[:, 0:2]; r_sh = kps[:, 3:5]
    mid = (l_sh + r_sh) / 2
    width = np.linalg.norm(l_sh - r_sh, axis=1, keepdims=True)
    width = np.where(width < 1e-6, 1.0, width)
    for i in range(FEATURE_DIM // 3):
        xi, yi = i * 3, i * 3 + 1
        kps[:, xi] = (kps[:, xi] - mid[:, 0]) / width[:, 0]
        kps[:, yi] = (kps[:, yi] - mid[:, 1]) / width[:, 0]
    return kps

def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_i = 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        frame_i += 1
        if frame_i % 2 != 0:
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = _holistic.process(rgb)
        frames.append(extract_frame_keypoints(results))
    cap.release()
    return normalize_keypoints(resample_frames(frames))

print('Faster extraction ready. Feature dim =', FEATURE_DIM)

Faster extraction ready. Feature dim = 144


In [ ]:
import time

def extract_all(entries, split_name):
    out_dir = f'{KEYPOINT_DIR}/{split_name}'
    os.makedirs(out_dir, exist_ok=True)

    total = len(entries)
    done, skipped, failed = 0, 0, 0
    t0 = time.time()

    for i, e in enumerate(entries):
        video_name = os.path.basename(e['video_rel']).replace('.mp4', '')
        out_path = f"{out_dir}/{e['gloss']}__{video_name}.npy"

        if os.path.exists(out_path):
            skipped += 1
            continue

        video_path = os.path.join(DATASET_DIR, e['gloss'], os.path.basename(e['video_rel']))
        if not os.path.exists(video_path):

            folder = os.path.join(DATASET_DIR, e['gloss'])
            target = os.path.basename(e['video_rel']).lower()
            match = None
            if os.path.isdir(folder):
                for f in os.listdir(folder):
                    if f.lower() == target:
                        match = os.path.join(folder, f)
                        break
            if match is None:
                failed += 1
                continue
            video_path = match

        try:
            kps = process_video(video_path)
            np.save(out_path, kps)

            done += 1
        except Exception as ex:
            print(f'  [fail] {video_path}: {ex}')
            failed += 1

        if (i + 1) % 50 == 0:
            elapsed = time.time() - t0
            rate = (done + skipped) / max(elapsed, 1)
            eta = (total - i - 1) / max(rate, 0.01) / 60
            print(f'  {split_name}: {i+1}/{total} | done={done} skip={skipped} fail={failed} | ETA {eta:.0f} min')

    print(f'✓ {split_name}: done={done}, skipped={skipped}, failed={failed}')


print('Extracting TRAIN split...')
extract_all(train_entries, 'train')
print('\nExtracting TEST split...')
extract_all(test_entries, 'test')

import json
with open(f'{OUTPUT_DIR}/label_to_gloss.json', 'w', encoding='utf-8') as f:
    json.dump(label_to_gloss, f, ensure_ascii=False, indent=2)
print('Saved label_to_gloss.json')

Extracting TRAIN split...
✓ train: done=0, skipped=4800, failed=0

Extracting TEST split...


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


  test: 450/1200 | done=42 skip=408 fail=0 | ETA 10 min
  test: 500/1200 | done=92 skip=408 fail=0 | ETA 18 min
  test: 550/1200 | done=142 skip=408 fail=0 | ETA 23 min
  test: 600/1200 | done=192 skip=408 fail=0 | ETA 26 min
  test: 650/1200 | done=242 skip=408 fail=0 | ETA 28 min
  test: 700/1200 | done=292 skip=408 fail=0 | ETA 29 min
  test: 750/1200 | done=342 skip=408 fail=0 | ETA 28 min
  test: 800/1200 | done=392 skip=408 fail=0 | ETA 27 min
  test: 850/1200 | done=442 skip=408 fail=0 | ETA 25 min
  test: 900/1200 | done=492 skip=408 fail=0 | ETA 22 min
  test: 950/1200 | done=542 skip=408 fail=0 | ETA 19 min
  test: 1000/1200 | done=592 skip=408 fail=0 | ETA 16 min
  test: 1050/1200 | done=642 skip=408 fail=0 | ETA 12 min
  test: 1100/1200 | done=692 skip=408 fail=0 | ETA 9 min
  test: 1150/1200 | done=742 skip=408 fail=0 | ETA 4 min
  test: 1200/1200 | done=792 skip=408 fail=0 | ETA 0 min
✓ test: done=792, skipped=408, failed=0
Saved label_to_gloss.json


In [ ]:
import glob
train_done = len(glob.glob(f'{KEYPOINT_DIR}/train/*.npy'))
test_done = len(glob.glob(f'{KEYPOINT_DIR}/test/*.npy'))
print(f'Train cached: {train_done}')
print(f'Test cached: {test_done}')

Train cached: 4800
Test cached: 1200


In [ ]:
import os
print(os.path.exists(f'{OUTPUT_DIR}/label_to_gloss.json'))

True


---
## Stage 3 — Load Keypoints + Augmentation

Fast — can re-run freely while tuning the model. Loads cached keypoints and builds
PyTorch datasets. Augmentation (mirroring, time-warp, jitter) expands the small
training set.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import glob
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

with open(f'{OUTPUT_DIR}/label_to_gloss.json', encoding='utf-8') as f:
    label_to_gloss = {int(k): v for k, v in json.load(f).items()}

glosses_sorted = sorted(set(label_to_gloss.values()))
gloss_to_idx = {g: i for i, g in enumerate(glosses_sorted)}
idx_to_gloss = {i: g for g, i in gloss_to_idx.items()}
NUM_CLASSES = len(gloss_to_idx)
print(f'Classes: {NUM_CLASSES}')

Device: cuda
Classes: 199


In [ ]:
def load_split(split_name):
    """Load all .npy keypoints for a split. Returns (X list, y list)."""
    files = glob.glob(f'{KEYPOINT_DIR}/{split_name}/*.npy')
    X, y = [], []
    for fp in files:
        gloss = os.path.basename(fp).split('__')[0]
        if gloss not in gloss_to_idx:
            continue
        X.append(np.load(fp))
        y.append(gloss_to_idx[gloss])
    return X, y

X_train, y_train = load_split('train')
X_test,  y_test  = load_split('test')
print(f'Train samples: {len(X_train)}')
print(f'Test samples : {len(X_test)}')

Train samples: 4776
Test samples : 1194


In [ ]:
FEATURE_DIM = 144
N_POINTS = FEATURE_DIM // 3

def aug_mirror(kps):
    """Horizontal mirror: flip x, swap left/right hand blocks."""
    k = kps.copy()
    for i in range(N_POINTS):
        k[:, i*3] = -k[:, i*3]
    lh = k[:, 6*3:27*3].copy()
    rh = k[:, 27*3:48*3].copy()
    k[:, 6*3:27*3] = rh
    k[:, 27*3:48*3] = lh
    return k

def aug_jitter(kps, sigma=0.01):
    """Add small Gaussian noise to coordinates."""
    noise = np.random.normal(0, sigma, kps.shape).astype(np.float32)
    return kps + noise

def aug_scale(kps, lo=0.9, hi=1.1):
    """Random uniform scale."""
    s = np.random.uniform(lo, hi)
    k = kps.copy()
    for i in range(N_POINTS):
        k[:, i*3]   *= s
        k[:, i*3+1] *= s
    return k

def aug_time_warp(kps):
    """Slightly speed up or slow down by resampling."""
    T = len(kps)
    factor = np.random.uniform(0.8, 1.2)
    new_T = max(5, int(T * factor))
    idx = np.linspace(0, T - 1, new_T).astype(int)
    warped = kps[idx]
    idx2 = np.linspace(0, new_T - 1, T).astype(int)
    return warped[idx2]


class SignDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X
        self.y = y
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        kps = self.X[i].copy()
        if self.augment:
            if np.random.rand() < 0.5:
                kps = aug_mirror(kps)
            if np.random.rand() < 0.4:
                kps = aug_scale(kps)
            if np.random.rand() < 0.4:
                kps = aug_time_warp(kps)
            if np.random.rand() < 0.25:
                kps = aug_jitter(kps, sigma=0.01)
        return torch.tensor(kps, dtype=torch.float32), self.y[i]


train_ds = SignDataset(X_train, y_train, augment=True)
test_ds  = SignDataset(X_test,  y_test,  augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)
print('Datasets ready.')

Datasets ready.


---
## Stage 4 — Bi-LSTM Model + Training

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, feature_dim=144, hidden=160, num_layers=2, num_classes=200, dropout=0.3):
        super().__init__()
        self.input_norm = nn.LayerNorm(feature_dim)
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.norm = nn.LayerNorm(hidden * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        x = self.input_norm(x)
        out, _ = self.lstm(x)
        pooled = out.mean(dim=1)
        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)
        return self.fc(pooled)

model = BiLSTMClassifier(hidden=160, num_classes=NUM_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 1,073,447


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)   # lighter smoothing
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 120
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8)

best_acc = 0.0
best_path = f'{OUTPUT_DIR}/bdsl_bilstm_best.pt'

def evaluate():
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for kps, labels in test_loader:
            kps = kps.to(device)
            labels = torch.tensor(labels).to(device) if not torch.is_tensor(labels) else labels.to(device)
            correct += (model(kps).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for kps, labels in train_loader:
        kps = kps.to(device)
        labels = torch.tensor(labels).to(device) if not torch.is_tensor(labels) else labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(kps), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    acc = evaluate()
    scheduler.step(acc)
    if acc > best_acc:
        best_acc = acc
        torch.save({'model_state': model.state_dict(), 'num_classes': NUM_CLASSES,
                    'idx_to_gloss': idx_to_gloss, 'feature_dim': 144, 'target_frames': 30}, best_path)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | loss {total_loss/len(train_loader):.3f} | acc {acc*100:.1f}% | best {best_acc*100:.1f}%')

print(f'\n✓ Best test accuracy: {best_acc*100:.1f}%')

Epoch   1/120 | loss 4.960 | acc 7.3% | best 7.3%
Epoch  10/120 | loss 1.868 | acc 48.2% | best 48.3%
Epoch  20/120 | loss 1.154 | acc 59.2% | best 59.2%
Epoch  30/120 | loss 0.938 | acc 61.0% | best 63.7%
Epoch  40/120 | loss 0.683 | acc 66.2% | best 66.2%
Epoch  50/120 | loss 0.625 | acc 66.8% | best 69.0%
Epoch  60/120 | loss 0.563 | acc 70.5% | best 70.5%
Epoch  70/120 | loss 0.544 | acc 70.4% | best 70.9%
Epoch  80/120 | loss 0.530 | acc 70.1% | best 71.5%
Epoch  90/120 | loss 0.524 | acc 71.5% | best 72.1%
Epoch 100/120 | loss 0.520 | acc 71.6% | best 72.2%
Epoch 110/120 | loss 0.514 | acc 71.2% | best 72.2%
Epoch 120/120 | loss 0.514 | acc 71.6% | best 72.7%

✓ Best test accuracy: 72.7%


In [ ]:
import torch

ckpt = torch.load(f'{OUTPUT_DIR}/bdsl_bilstm_best.pt', map_location='cpu')

cpu_model = BiLSTMClassifier(hidden=160, num_classes=NUM_CLASSES)
cpu_model.load_state_dict(ckpt['model_state'])
cpu_model.eval()
cpu_model = cpu_model.cpu()

example = torch.randn(1, 30, 144)
traced = torch.jit.trace(cpu_model, example)
traced.save(f'{OUTPUT_DIR}/bdsl_bilstm_scripted.pt')

import json
with open(f'{OUTPUT_DIR}/bdsl_idx_to_gloss.json', 'w', encoding='utf-8') as f:
    json.dump(idx_to_gloss, f, ensure_ascii=False, indent=2)

print('Re-exported model traced on CPU — will run on CPU server')

Re-exported model traced on CPU — will run on CPU server


---
## Stage 5 — Evaluate + Export for FastAPI Server

In [ ]:
import torch, glob, numpy as np

m = torch.jit.load(f'{OUTPUT_DIR}/bdsl_bilstm_scripted.pt', map_location='cpu')
m.eval()

correct = 0
total = 0
for gloss in ['maa', 'baba', 'valo', 'bhai', 'pakhi','mosha']:
    files = glob.glob(f'{KEYPOINT_DIR}/test/{gloss}__*.npy')
    for fp in files[:3]:
        kps = np.load(fp)
        x = torch.tensor(kps, dtype=torch.float32).unsqueeze(0)
        pred_idx = m(x).argmax(1).item()
        pred_gloss = idx_to_gloss[pred_idx]
        ok = pred_gloss == gloss
        correct += ok
        total += 1
        print(f'{gloss:12} → predicted {pred_gloss:12} {"✓" if ok else "✗"}')

print(f'\n{correct}/{total} correct on this sample')

In [ ]:
from collections import defaultdict

model.eval()
class_correct = defaultdict(int)
class_total = defaultdict(int)
with torch.no_grad():
    for kps, labels in test_loader:
        kps = kps.to(device)
        labels_t = torch.tensor(labels) if not torch.is_tensor(labels) else labels
        preds = model(kps).argmax(1).cpu()
        for p, l in zip(preds, labels_t):
            class_total[l.item()] += 1
            if p == l:
                class_correct[l.item()] += 1

accs = [(idx_to_gloss[c], class_correct[c]/class_total[c]) for c in class_total]
accs.sort(key=lambda x: x[1])
print('10 hardest signs:')
for gloss, a in accs[:10]:
    print(f'  {gloss:20} {a*100:.0f}%')
print('\n10 easiest signs:')
for gloss, a in accs[-10:]:
    print(f'  {gloss:20} {a*100:.0f}%')

In [ ]:
ckpt = torch.load(best_path)
model.load_state_dict(ckpt['model_state'])
model.eval()

example = torch.randn(1, 30, 144).to(device)
traced = torch.jit.trace(model, example)
traced.save(f'{OUTPUT_DIR}/bdsl_bilstm_scripted.pt')

with open(f'{OUTPUT_DIR}/bdsl_idx_to_gloss.json', 'w', encoding='utf-8') as f:
    json.dump(idx_to_gloss, f, ensure_ascii=False, indent=2)

print('Exported:')
print(f'  {OUTPUT_DIR}/bdsl_bilstm_best.pt        (full checkpoint)')
print(f'  {OUTPUT_DIR}/bdsl_bilstm_scripted.pt    (TorchScript for serving)')
print(f'  {OUTPUT_DIR}/bdsl_idx_to_gloss.json     (label map)')
print('\nThese are saved on your Drive and ready for the FastAPI server.')

In [ ]:
import os
for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.pt') or f.endswith('.json'):
        print(f, f'({os.path.getsize(os.path.join(OUTPUT_DIR, f))/1024:.0f} KB)')


label_to_gloss.json (4 KB)
bdsl_bilstm_best.pt (4201 KB)
bdsl_idx_to_gloss.json (4 KB)
bdsl_bilstm_scripted.pt (4212 KB)
